# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
import pandas as pd
from pathlib import Path
import subprocess

repo_path = Path("/content/flyrank-ml-workspace2")

if not repo_path.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/suyashverma0/flyrank-ml-workspace2.git",
        str(repo_path)
    ], check=True)

matches = list(repo_path.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError("Dataset CSV nahi mili.")

data_path = matches[0]
df = pd.read_csv(data_path)

print("Dataset loaded:", df.shape)
df.head()

Dataset loaded: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
import pandas as pd
import numpy as np

# Create target
df["is_declining_proxy"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Select features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent"
]

feature_columns = numeric_features + categorical_features

# Create X, y and groups
X = df[feature_columns].copy()
y = df["is_declining_proxy"].copy()
groups = df["client_id"].copy()

# Convert numeric columns
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups:", groups.nunique())

X shape: (30000, 10)
y shape: (30000,)
Groups: 32


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Verify available data

required_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

print("Column availability:")
for col in required_columns:
    print(col, ":", col in df.columns)

print("\nSearch volume vs impressions correlation:")

correlation = df[
    ["search_volume", "impressions_90d"]
].corr().iloc[0, 1]

print("Correlation:", round(correlation, 4))

print("\nRows available for position analysis:")

valid_rows = df[
    ["clicks_90d", "impressions_90d", "avg_position"]
].notna().all(axis=1).sum()

print("Valid rows:", valid_rows)

Column availability:
search_volume : True
impressions_90d : True
clicks_90d : True
avg_position : True

Search volume vs impressions correlation:
Correlation: 0.0012

Rows available for position analysis:
Valid rows: 30000


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [13]:
def precision_at_k(order, labels, k):
    selected = labels.loc[order[:min(k, len(order))]]
    return selected.mean()

print("precision_at_k function created successfully!")

precision_at_k function created successfully!


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Logistic Regression model
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Model created successfully!")

Model created successfully!


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.base import clone
import numpy as np

# Function to evaluate one split
def evaluate_model_split(train_idx, test_idx, split_name):

    X_tr = X.iloc[train_idx].copy()
    X_te = X.iloc[test_idx].copy()

    y_tr = y.iloc[train_idx].copy()
    y_te = y.iloc[test_idx].copy()

    current_model = clone(model)
    current_model.fit(X_tr, y_tr)

    probabilities = current_model.predict_proba(X_te)[:, 1]

    ranking = pd.Series(
        probabilities,
        index=X_te.index
    ).sort_values(ascending=False).index

    return {
        "Split": split_name,
        "Train Rows": len(train_idx),
        "Test Rows": len(test_idx),
        "Precision@10": precision_at_k(ranking, y_te, 10),
        "Precision@50": precision_at_k(ranking, y_te, 50),
        "Precision@100": precision_at_k(ranking, y_te, 100),
        "Test Base Rate": y_te.mean()
    }


# BEFORE: Random row split
indices = np.arange(len(X))

random_train_idx, random_test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_result = evaluate_model_split(
    random_train_idx,
    random_test_idx,
    "Random Row Split"
)


# AFTER: Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

group_train_idx, group_test_idx = next(
    splitter.split(X, y, groups=groups)
)

group_result = evaluate_model_split(
    group_train_idx,
    group_test_idx,
    "Client Grouped Split"
)


# Check client overlap
random_overlap = len(
    set(groups.iloc[random_train_idx])
    & set(groups.iloc[random_test_idx])
)

group_overlap = len(
    set(groups.iloc[group_train_idx])
    & set(groups.iloc[group_test_idx])
)

print("Random split client overlap:", random_overlap)
print("Grouped split client overlap:", group_overlap)

assert group_overlap == 0

# Comparison table
validation_results = pd.DataFrame([
    random_result,
    group_result
])

display(validation_results.round(4))

Random split client overlap: 31
Grouped split client overlap: 0


,Split,Train Rows,Test Rows,Precision@10,Precision@50,Precision@100,Test Base Rate
0,Random Row Split,22500,7500,0.7,0.70,0.66,0.5420
1,Client Grouped Split,22885,7115,0.6,0.64,0.55,0.5165


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature list from Week-5
final_features = numeric_features + categorical_features

# Forbidden columns
forbidden_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "is_declining_label"
]

# Performance fields excluded from metadata-only model
performance_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Leakage report
leakage_report = pd.DataFrame({
    "Feature": final_features,
    "Forbidden": [
        col in forbidden_columns
        for col in final_features
    ],
    "Performance Field": [
        col in performance_columns
        for col in final_features
    ]
})

display(leakage_report)

# Assertions
assert not set(final_features).intersection(forbidden_columns)
assert not set(final_features).intersection(performance_columns)

assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("Total features:", len(final_features))
print("Forbidden feature overlap:",
      len(set(final_features).intersection(forbidden_columns)))

print("Performance feature overlap:",
      len(set(final_features).intersection(performance_columns)))

print("Leakage audit: PASSED")

,Feature,Forbidden,Performance Field
0,search_volume,False,False
1,competition,False,False
2,cpc,False,False
3,word_count,False,False
4,char_count,False,False
5,content_age_days,False,False
6,days_since_last_update,False,False
7,competition_level,False,False
8,content_type,False,False
9,main_intent,False,False


Total features: 10
Forbidden feature overlap: 0
Performance feature overlap: 0
Leakage audit: PASSED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_audit = pd.DataFrame([
    {
        "Type": "Original Claim",
        "Statement": (
            "The Logistic Regression model can identify declining "
            "content and predict which pages need refreshing."
        )
    },
    {
        "Type": "Rewritten Claim",
        "Statement": (
            "The model ranks content items by their estimated likelihood "
            "of matching a historical declining-content proxy. "
            "Its client-grouped Precision@K supports decision-making "
            "for further content review, not proof of refresh success."
        )
    }
])

display(claim_audit)

,Type,Statement
0,Original Claim,The Logistic Regression model can identify dec...
1,Rewritten Claim,The model ranks content items by their estimat...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.